In [1]:
"""
Test TypeInferenceEngine.get_type() method
Copy and paste this into a Jupyter notebook cell
"""

import ast
from analyzer.analysis.expression_traversal import TypeInferenceEngine
from analyzer.analysis import Scope

# Create a mock project node (not used for literals/simple lookups yet)
class MockProjectNode:
    pass

project = MockProjectNode()
engine = TypeInferenceEngine(project)

# Create a scope with some test variables
scope = Scope()
scope.push_frame()
scope.add("x", "int")
scope.add("name", "str")
scope.add("is_valid", "bool")

print("=" * 70)
print("TypeInferenceEngine.get_type() Test Results")
print("=" * 70)

# Test cases
test_cases = [
    # Literals
    ("42", "Literal integer"),
    ("3.14", "Literal float"),
    ('"hello"', "Literal string"),
    ("True", "Literal boolean"),
    ("False", "Literal boolean"),
    ("None", "Literal None"),
    
    # Variable lookups (from scope)
    ("x", "Variable lookup (int)"),
    ("name", "Variable lookup (str)"),
    ("is_valid", "Variable lookup (bool)"),
    ("unknown_var", "Unknown variable (not in scope)"),
]

for expr_string, description in test_cases:
    print(f"\n{description}")
    print(f"Expression: {expr_string}")
    print("-" * 70)
    
    # Parse the expression
    expr_ast = ast.parse(expr_string, mode='eval').body
    
    # Get the type
    type_fqn = engine.get_type(expr_ast, scope)
    
    # Display result
    if type_fqn:
        print(f"✓ Type: {type_fqn}")
    else:
        print(f"✗ Type: None (could not determine)")

print("\n" + "=" * 70)
print("Test Complete!")
print("=" * 70)

TypeInferenceEngine.get_type() Test Results

Literal integer
Expression: 42
----------------------------------------------------------------------
✓ Type: int

Literal float
Expression: 3.14
----------------------------------------------------------------------
✓ Type: float

Literal string
Expression: "hello"
----------------------------------------------------------------------
✓ Type: str

Literal boolean
Expression: True
----------------------------------------------------------------------
✓ Type: bool

Literal boolean
Expression: False
----------------------------------------------------------------------
✓ Type: bool

Literal None
Expression: None
----------------------------------------------------------------------
✓ Type: NoneType

Variable lookup (int)
Expression: x
----------------------------------------------------------------------
✓ Type: int

Variable lookup (str)
Expression: name
----------------------------------------------------------------------
✓ Type: str

Varia

In [2]:
"""
Test ModuleAnalysisVisitor with Type Inference
Copy and paste this into a Jupyter notebook cell
"""

import ast
from analyzer.analysis.visitors.module_analysis_visitor import ModuleAnalysisVisitor

# Create a simple test module with various assignments
test_code = """
# Literal assignments
x = 42
name = "Alice"
pi = 3.14
is_valid = True
result = None

# Variable-to-variable assignment (should lookup in scope)
y = x
greeting = name

# Annotated assignment
count: int = 100
"""

# Parse the test code
module_ast = ast.parse(test_code)

# Create a mock module node
class MockModuleNode:
    def __init__(self, ast_module):
        self.source_data = type('obj', (object,), {'ast_node': ast_module})()
        self.name = "test_module"
    
    def get_project(self):
        # Return mock project node
        class MockProjectNode:
            pass
        return MockProjectNode()

# Create the module node
module_node = MockModuleNode(module_ast)

# Create and run the visitor
print("=" * 70)
print("ModuleAnalysisVisitor Type Inference Test")
print("=" * 70)
print()

visitor = ModuleAnalysisVisitor(module_node)
visitor.visit(module_ast)

print()
print("=" * 70)
print("Final Scope Contents:")
print("=" * 70)

# Display what's in the scope
if visitor.scope._frames:
    frame = visitor.scope._frames[0]
    if frame._bindings:
        for var_name, type_fqn in frame._bindings.items():
            print(f"  {var_name}: {type_fqn}")
    else:
        print("  (empty)")
else:
    print("  (no frames)")

print()
print(f"Total assignments processed: {visitor.assignment_count}")
print()
print("=" * 70)
print("Test Complete!")
print("=" * 70)

ModuleAnalysisVisitor Type Inference Test

   Inferred: x = int (line 3)
   Inferred: name = str (line 4)
   Inferred: pi = float (line 5)
   Inferred: is_valid = bool (line 6)
   Inferred: result = NoneType (line 7)
   Inferred: y = int (line 10)
   Inferred: greeting = str (line 11)
   Inferred: count: ... = int (line 14)

Final Scope Contents:
  x: int
  name: str
  pi: float
  is_valid: bool
  result: NoneType
  y: int
  greeting: str
  count: int

Total assignments processed: 8

Test Complete!


In [4]:
"""
Simple test for dot() navigation without using full atlas.py
"""

import sys
sys.path.insert(0, r'C:\Users\thoma\projects\PhoenixMegaproject\atlas')

from analyzer.nodes import ProjectNode
from analyzer.reconnaissance.discovery import discover_project_structure

# Build project directly
print("Building project...")
structure = discover_project_structure("sample_files")
project = ProjectNode(structure)

print("\n" + "="*60)
print("Testing dot() Navigation with Canonical Names")
print("="*60)

# Navigate to a module
print("\n1. Navigate to module:")
core_module = project.dot("core")
print(f"   project.dot('core') = {core_module}")

if core_module:
    # Navigate to a class
    print("\n2. Navigate to class:")
    # Try to find what classes exist
    classes = core_module.list_classes()
    print(f"   Available classes: {[c.name for c in classes]}")
    
    if classes:
        first_class = classes[0]
        print(f"   Using first class: {first_class.name}")
        
        # Navigate to a method
        print("\n3. Navigate to method:")
        methods = first_class.list_methods()
        print(f"   Available methods: {[m.name for m in methods]}")
        
        if methods:
            first_method = methods[0]
            print(f"   Using first method: {first_method.name}")
            
            # Navigate to return using canonical name
            print("\n4. Navigate to return (canonical name):")
            return_node = first_method.dot("return")
            print(f"   method.dot('return') = {return_node}")
            
            if return_node:
                print(f"   Return node name: {return_node.name}")
                
                # Navigate to type using canonical name  
                print("\n5. Navigate to type (canonical name):")
                type_node = return_node.dot("type")
                print(f"   return_node.dot('type') = {type_node}")
                
                if type_node:
                    print(f"   ✅ Type node name: {type_node.name}")
                    print(f"   Type node source: {type_node.source_data}")
                    
                    # Try to get the actual type string
                    import ast
                    try:
                        type_string = ast.unparse(type_node.source_data)
                        print(f"   ✅ Actual type: {type_string}")
                    except:
                        print(f"   (Could not unparse type)")
                else:
                    print("   ⚠️  No type annotation on return")
            
            # Test argument navigation
            print("\n6. Navigate to argument:")
            args = first_method.list_arguments()
            print(f"   Arguments: {[a.name for a in args]}")
            
            if args:
                first_arg = args[0]
                print(f"   First argument: {first_arg.name}")
                arg_type = first_arg.dot("type")
                if arg_type:
                    print(f"   ✅ first_arg.dot('type') = {arg_type}")
                    print(f"   Type node name: {arg_type.name}")
                    import ast
                    try:
                        arg_type_string = ast.unparse(arg_type.source_data)
                        print(f"   ✅ Actual type: {arg_type_string}")
                    except:
                        print(f"   (Could not unparse type)")
                else:
                    print("   ⚠️  No type annotation on argument")

# Test navigation failure (returns None)
print("\n7. Test navigation failure:")
nonexistent = project.dot("DoesNotExist")
print(f"   project.dot('DoesNotExist') = {nonexistent}")
print(f"   ✅ Returns None as expected")

print("\n" + "="*60)
print("✅ dot() Navigation Tests Complete!")
print("="*60)

Building project...

Testing dot() Navigation with Canonical Names

1. Navigate to module:
   project.dot('core') = Package(core)

2. Navigate to class:
   Available classes: ['BaseEntity', 'ConfigurableEntity', 'ValidationError', 'AuthenticationError', 'UtilityHelper']
   Using first class: BaseEntity

3. Navigate to method:
   Available methods: ['__init__', 'get_id', 'get_name', 'update_metadata', 'has_metadata', 'to_dict']
   Using first method: __init__

4. Navigate to return (canonical name):
   method.dot('return') = Return(return)
   Return node name: return

5. Navigate to type (canonical name):
   return_node.dot('type') = None
   ⚠️  No type annotation on return

6. Navigate to argument:
   Arguments: ['self', 'entity_id', 'name', 'metadata']
   First argument: self
   ⚠️  No type annotation on argument

7. Test navigation failure:
   project.dot('DoesNotExist') = None
   ✅ Returns None as expected

✅ dot() Navigation Tests Complete!
